# CS156 · Session 6 — Logistic Regression and Maximum Likelihood Estimation

**Based on the supplied Session 6 PCW ("Generated Forum Code Workbook", pp. 1–4).** This notebook reorganizes the PCW's preclass questions, synthetic *Mujhe Chahiye Bollywood* case study, likelihood comparison, and gradient extension into runnable cells. Explanatory derivations and a numerical MLE fit are clearly marked as additional learning material.

**Learning goals:** interpret logistic-regression parameters; derive Bernoulli likelihood and log-likelihood; compare candidate parameters; compute and check gradients; find maximum-likelihood estimates.

Run cells **top to bottom**. The experiment is entirely synthetic—no external dataset download or JAX installation is required.

## 0. Preclass review: Gaussian maximum likelihood (additional background)

The PCW opens with an interview prompt asking you to explain how to fit a Gaussian probability density function. For $x_1,\dots,x_n\overset{\text{iid}}{\sim}\mathcal N(\mu,\sigma^2)$,

$$
p(x\mid\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).
$$

The unknown parameters are $\mu$ (mean) and $\sigma^2$ (variance). Maximizing the joint likelihood of independent observations gives

$$
\widehat\mu=\frac1n\sum_{i=1}^n x_i,\qquad
\widehat{\sigma}^2=\frac1n\sum_{i=1}^n (x_i-\widehat\mu)^2.
$$

The denominator $n$ in the variance formula is the **maximum-likelihood** estimator; the unbiased sample-variance formula uses $n-1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

RNG = np.random.default_rng(42)

# Three observations for the Gaussian-MLE review (illustrative data).
gaussian_example = np.array([1.5, 2.0, 4.0])
mu_hat = gaussian_example.mean()
variance_mle = np.mean((gaussian_example - mu_hat) ** 2)
print(f"Gaussian example: mean MLE={mu_hat:.3f}, variance MLE={variance_mle:.3f}")

## 1. PCW Question 1 — Logistic regression

**Scenario (synthetic):** a student wants to estimate the probability of securing a movie role from the amount of training they complete. Following the PCW, $x$ is **effort measured in thousands of hours**, while $y\in\{0,1\}$ says whether the student gets a role.

$$
P(y=1\mid x;\beta_0,\beta_1)=\sigma(\beta_0+\beta_1x),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

- $\beta_0$ is the **intercept**: the log-odds when $x=0$.
- $\beta_1$ is the **slope**: each additional 1,000 hours adds $\beta_1$ to the log-odds.
- $\beta_0,\beta_1$ are learnable **parameters**; $x_i$ are observed input values, not parameters.
- The binary output $y_i$ is an observed result. The model produces a predicted **probability** $p_i=P(y_i=1\mid x_i)$.

The sigmoid is nonlinear as a function of $x$, but its **log-odds** are linear:

$$\log\frac{p_i}{1-p_i}=\beta_0+\beta_1x_i.$$

## 2. PCW Question 2 — Likelihood and log-likelihood

The PCW first considers a coin with independent observations heads, tails, heads, heads, tails. If heads has probability $p$, then

$$L(p)=p(1-p)\,p\,p(1-p)=p^3(1-p)^2,$$
$$\ell(p)=\log L(p)=3\log p+2\log(1-p).$$

For logistic regression, the observations are **conditionally independent Bernoulli variables** with different success probabilities $p_i=\sigma(\beta_0+\beta_1 x_i)$:

$$
L(\beta_0,\beta_1)=\prod_{i=1}^n p_i^{y_i}(1-p_i)^{1-y_i}.
$$

Taking the logarithm turns the product into a sum, which is easier and more numerically stable to compute:

$$
\ell(\beta_0,\beta_1)=\sum_{i=1}^n\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].
$$

**MLE maximizes $\ell$; minimizing binary cross-entropy is equivalent to minimizing $-\ell/n$.**

## 3. PCW core question — Generate the synthetic data

The source PCW generates **1,000** examples with $x$ drawn uniformly between 0 and 3 (thousands of hours), and hidden generating parameters $\beta_0=-5$, $\beta_1=3$. We use the same model and a fixed seed for reproducibility; the randomly generated values will not exactly match the screenshot's results.

In [ ]:
NUM_DATAPOINTS = 1_000
TRUE_BETA0 = -5.0
TRUE_BETA1 = 3.0


def sigmoid(z):
    """Overflow-safe sigmoid for scalars and NumPy arrays."""
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    positive = z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    ez = np.exp(z[~positive])
    out[~positive] = ez / (1.0 + ez)
    return out


x_data = RNG.uniform(0.0, 3.0, size=NUM_DATAPOINTS)
prob_true = sigmoid(TRUE_BETA0 + TRUE_BETA1 * x_data)
y_data = RNG.binomial(1, prob_true).astype(int)

x_curve = np.linspace(0, 3, 250)
plt.figure(figsize=(8, 4.5))
plt.scatter(x_data, y_data, s=7, alpha=0.15, label="Observed outcome")
plt.plot(x_curve, sigmoid(TRUE_BETA0 + TRUE_BETA1 * x_curve),
         label="Generating probability")
plt.xlabel("Effort (thousands of hours)")
plt.ylabel("Role secured / predicted probability")
plt.yticks([0, 0.5, 1]); plt.legend(); plt.tight_layout(); plt.show()

## 4. PCW core question — Compare candidate log-likelihoods

The PCW compares **Ashwarya's** $(\beta_0,\beta_1)=(3,-3)$ against **Arun's** $(4,-1)$, using the **same generated data**. Calculate each candidate's log-likelihood and identify the larger value. Log-likelihoods are usually negative, so "larger" means *less negative*.

For numerical stability, compute the Bernoulli log-likelihood from the **logits** $z_i=\beta_0+\beta_1x_i$:

$$\ell=\sum_i [y_i z_i-\log(1+e^{z_i})].$$

In [ ]:
def log_likelihood(beta0, beta1, xs, ys):
    logits = beta0 + beta1 * np.asarray(xs, dtype=float)
    labels = np.asarray(ys, dtype=float)
    return float(np.sum(labels * logits - np.logaddexp(0.0, logits)))


candidates = {"Ashwarya (3, -3)": (3.0, -3.0),
              "Arun (4, -1)": (4.0, -1.0),
              "Generating (-5, 3)": (TRUE_BETA0, TRUE_BETA1)}
ll_values = {name: log_likelihood(b0, b1, x_data, y_data)
             for name, (b0, b1) in candidates.items()}
for name, ll in ll_values.items():
    print(f"{name:<25} log-likelihood: {ll:,.2f}")
print("Higher of the two proposed candidates:",
      max(list(candidates)[:2], key=ll_values.get))

## 5. PCW extension — Gradient of the log-likelihood

The source PCW's gradient extension can be implemented with NumPy, so installing JAX is **optional**. With $p_i=\sigma(\beta_0+\beta_1x_i)$,

$$
\frac{\partial\ell}{\partial\beta_0}=\sum_i(y_i-p_i),\qquad
\frac{\partial\ell}{\partial\beta_1}=\sum_i x_i(y_i-p_i).
$$

The gradient points toward **increasing** log-likelihood. Evaluate it at Ashwarya's initial parameters, as requested in the PCW, and check it against a numerical finite-difference approximation.

In [ ]:
def gradient_log_likelihood(beta0, beta1, xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    errors = ys - sigmoid(beta0 + beta1 * xs)
    return np.array([errors.sum(), np.dot(errors, xs)])


ashwarya_gradient = gradient_log_likelihood(3.0, -3.0, x_data, y_data)
print("Gradient at (3, -3):", np.round(ashwarya_gradient, 3))

# Additional check beyond the original PCW.
eps = 1e-5
finite_diff = np.array([
    (log_likelihood(3.0 + eps, -3.0, x_data, y_data)
     - log_likelihood(3.0 - eps, -3.0, x_data, y_data)) / (2 * eps),
    (log_likelihood(3.0, -3.0 + eps, x_data, y_data)
     - log_likelihood(3.0, -3.0 - eps, x_data, y_data)) / (2 * eps)
])
assert np.allclose(ashwarya_gradient, finite_diff, rtol=1e-5, atol=1e-4)
print("Finite-difference gradient check: passed")

## 6. Additional experiment — Optimize the parameters (MLE)

The PCW focuses on comparing guesses and finding the gradient. The following optional extension actually **estimates** the parameters by maximizing the log-likelihood with BFGS. Since `scipy.optimize.minimize` minimizes functions, we minimize the *negative* log-likelihood and supply its negative gradient.

In [ ]:
def negative_log_likelihood(beta):
    return -log_likelihood(beta[0], beta[1], x_data, y_data)


def negative_gradient(beta):
    return -gradient_log_likelihood(beta[0], beta[1], x_data, y_data)


fit = minimize(negative_log_likelihood, x0=np.array([0.0, 0.0]),
               jac=negative_gradient, method="BFGS")
if not fit.success:
    print("Optimizer note:", fit.message)
beta0_hat, beta1_hat = fit.x
print(f"Estimated (beta0, beta1): ({beta0_hat:.3f}, {beta1_hat:.3f})")
print(f"Generating (beta0, beta1): ({TRUE_BETA0:.3f}, {TRUE_BETA1:.3f})")
print(f"MLE log-likelihood: {log_likelihood(*fit.x, x_data, y_data):.2f}")

plt.figure(figsize=(8, 4))
plt.scatter(x_data, y_data, s=6, alpha=0.12, label="Observed outcome")
plt.plot(x_curve, sigmoid(TRUE_BETA0 + TRUE_BETA1*x_curve), label="Generating model")
plt.plot(x_curve, sigmoid(beta0_hat + beta1_hat*x_curve),
         linestyle="--", label="MLE fit")
plt.xlabel("Effort (thousands of hours)")
plt.ylabel("Probability of securing role")
plt.legend(); plt.tight_layout(); plt.show()

## 7. Additional experiment — 95% probability threshold

For $\beta_1>0$, solve $\sigma(\beta_0+\beta_1x)=0.95$:

$$
x_{95}=\frac{\log(0.95/0.05)-\beta_0}{\beta_1}.
$$

Because the PCW's data are **invented**, the answer below is a result of a toy model, not real career advice. Also remember that $x$ is measured in **thousands of hours**.

In [ ]:
if beta1_hat > 0:
    x95 = (np.log(0.95 / 0.05) - beta0_hat) / beta1_hat
    print(f"Estimated 95% threshold: {x95:.3f} thousand hours "
          f"({x95*1000:,.0f} hours)")
else:
    print("The fitted slope is nonpositive; the increasing-effort threshold interpretation does not apply.")

### Session 6 reflection

1. Why do we maximize **log**-likelihood rather than multiply hundreds of probabilities directly?
2. Why are $\beta_0$ and $\beta_1$ parameters, but $x_i$ and $y_i$ data?
3. What does the sign of $\partial\ell/\partial\beta_1$ tell us at a proposed estimate?
4. Why can different random samples produce different MLEs even if the generating parameters do not change?

*Source: supplied `Session 6.pdf`, PCW pp. 1–4. The Gaussian worked example, finite-difference check, BFGS fit, and explicit 95% calculation are study additions.*